# BMIN 5200 — Week 9 in-class exercise
## Entropy, information gain, and a tree you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week09.ipynb)

**Time:** ~25 minutes · **Pairs with:** Information theory & machine learning

### What you'll do
- Compute entropy, conditional entropy, and mutual information by hand from the definitions on the slides, not from a library
- Rank six candidate predictors of 30-day readmission by how much they actually tell you about the outcome
- Fit a shallow decision tree and confirm it splits first on exactly the feature your mutual information calculation picked
- Read the tree out as IF/THEN rules, and watch an unrestricted tree memorize the training set

### Why it matters
Every risk model you have ever been shown was built by some procedure for deciding which variable to look at first. Information theory gives that decision a unit — the bit — so "hemoglobin is more informative than insurance type" stops being a hunch and becomes a number you can audit. And a shallow decision tree is the one machine-learned model whose reasoning a clinician can read off the page and argue with, which is why it keeps reappearing in bedside scoring systems.

Setup. Everything here is preinstalled in Colab, so there is nothing to install. The seed is fixed so that every laptop in the room prints identical numbers and Joe can say a specific value out loud.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.feature_selection import mutual_info_classif

rng = np.random.default_rng(5200)
pd.set_option("display.width", 120)

## The cohort

This is a **synthetic** 30-day readmission cohort — 1,200 fictional discharges, no real patient
data. We generate it inline rather than downloading it so that the ground truth is known to us:
readmission risk really is driven by prior admissions, age, hemoglobin, and creatinine, and it
really is *not* driven by how far the patient lives from the hospital.

Read the generator. Note the last few lines, where `prior_admissions` is recorded only
partially for patients whose care is split with a community partner network. That is a
deliberate flaw, it is realistic, and we are going to spend Weeks 11 and 12 finding it.
**The same function, with the same seed, appears in the Week 11 and Week 12 notebooks**, so the
model you build today is the model you will explain and audit later in the semester.

In [ ]:
def make_readmission_cohort(n_patients=1200, seed=5200):
    """Synthetic 30-day readmission cohort. Not real patient data.

    Identical in the Week 9, Week 11, and Week 12 notebooks — same code, same seed — so all
    three weeks work on exactly the same 1,200 discharges. Nothing is saved to disk between
    notebooks; each one regenerates the cohort inline. The function builds its own generator
    so the cohort does not depend on what else has drawn from `rng` above it.
    """
    rng = np.random.default_rng(seed)

    care_network = rng.choice(["in_network", "community_partner"],
                              size=n_patients, p=[0.68, 0.32])
    partner = care_network == "community_partner"

    age = np.clip(rng.normal(68, 12, n_patients), 40, 95).round(0)
    true_prior_admissions = rng.poisson(1.8, n_patients)
    hemoglobin = np.clip(rng.normal(12.4, 1.6, n_patients)
                         - 0.9 * (true_prior_admissions > 2), 7.0, 17.0)
    creatinine = np.clip(rng.lognormal(np.log(1.0), 0.32, n_patients), 0.4, 6.0)

    # Partner-network patients are likelier to be on Medicaid and to live farther out.
    ins_probs = np.where(partner[:, None],
                         np.array([0.18, 0.34, 0.48]),
                         np.array([0.52, 0.36, 0.12]))
    draw = rng.random(n_patients)
    insurance_type = np.array(["commercial", "medicare", "medicaid"])[
        (draw[:, None] > ins_probs.cumsum(axis=1)).sum(axis=1)]

    distance_from_hospital = np.round(
        np.clip(rng.gamma(2.0, np.where(partner, 9.0, 3.4)), 0.5, 90.0), 1)

    # Ground truth: risk is driven by the TRUE admission history, identically in both groups,
    # and never by distance from the hospital.
    log_odds = (-2.0
                + 0.62 * true_prior_admissions
                + 0.030 * (age - 68)
                - 0.26 * (hemoglobin - 12.4)
                + 0.55 * (creatinine - 1.0))
    risk = 1.0 / (1.0 + np.exp(-log_odds))
    readmitted_30d = (rng.random(n_patients) < risk).astype(int)

    # The planted flaw: only 45% of a partner-network patient's prior admissions reach our
    # chart, against 97% for patients who stay in network. The illness is the same; the
    # documentation is not.
    capture = np.where(partner, 0.45, 0.97)
    recorded_prior = rng.binomial(true_prior_admissions, capture)

    return pd.DataFrame({
        "age": age.astype(int),
        "prior_admissions": recorded_prior,
        "hemoglobin": hemoglobin.round(1),
        "creatinine": creatinine.round(2),
        "insurance_type": insurance_type,
        "distance_from_hospital": distance_from_hospital,
        "care_network": care_network,
        "readmitted_30d": readmitted_30d,
    })


cohort = make_readmission_cohort()
print(cohort.head(6).to_string(index=False))
print(f"\n{len(cohort)} discharges, {cohort.readmitted_30d.mean():.1%} readmitted within 30 days")

## Part 1 — Entropy, from the definition

Entropy is $H(X) = -\sum_x p(x)\log_2 p(x)$, and with $\log_2$ the unit is bits. The slide
"Interpretations of H(X)" gives you the reading that matters clinically: entropy is the average
number of yes/no questions you need to pin down the answer, so it is a measure of how much you
do not yet know. A perfectly balanced coin costs 1 bit; an outcome you can already predict costs
0 bits. Run this and check the three sanity values before we use the function on anything real.

In [ ]:
def entropy(labels):
    """H(X) in bits, straight from the definition. No library call."""
    labels = np.asarray(labels)
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / counts.sum()
    # abs() is here only so a degenerate distribution prints 0.0000 rather than -0.0000.
    return float(abs(-(probabilities * np.log2(probabilities)).sum()))


print(f"fair coin              H = {entropy([0, 1]):.4f} bits")
print(f"nothing ever happens   H = {entropy([0, 0, 0, 0]):.4f} bits")
print(f"1-in-4 event           H = {entropy([0, 0, 0, 1]):.4f} bits")

outcome = cohort["readmitted_30d"]
h_outcome = entropy(outcome)
print(f"\n30-day readmission     H = {h_outcome:.4f} bits "
      f"(base rate {outcome.mean():.1%})")

So before we look at a single lab value we are 0.92 bits ignorant about whether a given patient
comes back. Every predictor we add is worth exactly as much as the number of those bits it
removes. That quantity has a name — mutual information — and the next part is where you compute it.

## Part 2 — Conditional entropy and mutual information

Conditional entropy is the *weighted* average of the outcome's entropy inside each group of
patients defined by a feature: $H(Y \mid X) = \sum_x p(x)\,H(Y \mid X{=}x)$. The weighting
matters — a bin holding four patients should not count as much as a bin holding four hundred.
Mutual information is then just what you started with minus what is left:
$I(X;Y) = H(Y) - H(Y \mid X)$, the bits of ignorance the feature actually removes.

These definitions are stated for discrete variables, so `bin_feature` below chops each continuous
lab into quartiles first. Fill in the one line marked TODO. The placeholder ignores the feature
entirely, which is why it will report 0.000 bits for everything.

In [ ]:
def bin_feature(column, n_bins=4):
    """Entropy is defined over discrete outcomes, so continuous labs become quartiles."""
    if column.dtype == object or column.nunique() <= 6:
        return column.astype(str)
    return pd.qcut(column, n_bins, labels=False, duplicates="drop").astype(str)


def conditional_entropy(feature, labels):
    """H(Y | X): the outcome entropy that survives after you know the feature."""
    feature, labels = np.asarray(feature), np.asarray(labels)
    total = len(labels)
    running = 0.0
    for value in np.unique(feature):
        in_bin = feature == value
        weight = in_bin.sum() / total
        # TODO: add this bin's contribution, weight * H(Y | X = value), to `running`.
        # Placeholder below pretends the feature told us nothing, so every I(X;Y) is 0.
        running += weight * entropy(labels)
    return running


candidates = ["age", "prior_admissions", "hemoglobin", "creatinine",
              "insurance_type", "distance_from_hospital", "care_network"]

ranking = []
for feature_name in candidates:
    binned = bin_feature(cohort[feature_name])
    h_given = conditional_entropy(binned, outcome)
    ranking.append({"feature": feature_name,
                    "H(Y|X)": round(h_given, 4),
                    "I(X;Y) bits": round(h_outcome - h_given, 4)})

ranking = pd.DataFrame(ranking).sort_values("I(X;Y) bits", ascending=False)
print(f"H(Y) = {h_outcome:.4f} bits before we know anything\n")
print(ranking.to_string(index=False))

Check your work against one number rather than eyeballing the whole table. With
`conditional_entropy` written correctly, `prior_admissions` should leave 0.8393 bits of
uncertainty behind, for a mutual information of 0.0814 bits. Run the cell below; if it says
"not yet", your bin contribution is probably unweighted.

In [ ]:
check = conditional_entropy(bin_feature(cohort["prior_admissions"]), outcome)
if abs(check - 0.8393) < 5e-4:
    print(f"H(Y | prior_admissions) = {check:.4f} bits — matches, carry on")
else:
    print(f"H(Y | prior_admissions) = {check:.4f} bits — not yet; expected 0.8393")
    print("Every group of patients has to be weighted by how many patients are in it.")

Two things are worth saying out loud about that ranking. First, the numbers are small: the best
single predictor removes under a tenth of a bit, which is an honest statement about how hard
readmission prediction is and about why these models rarely beat a good discharge nurse.
Second, `distance_from_hospital` is near the bottom, which is correct — we built it to have no
causal effect on the outcome. Hold onto that fact. In Week 11 an importance measure is going to
tell you the opposite.

## Predict before you run

You now have a ranking of features by bits of information gained. A decision tree, per the
"Tree Induction" and "Decision Tree: Splitting by..." slides, greedily picks the split that
maximizes information gain at every node — so its very first split is a decision made on
exactly the quantity you just computed.

**Commit to an answer before running the next cell.** Which feature will `sklearn` put at the
root of the tree, and what threshold will it choose? Say it out loud to the person next to you.

## Part 3 — The tree does the arithmetic you just did

`DecisionTreeClassifier` is capped at depth 3 here so the whole thing fits on a slide. We hold
out 30% of the cohort for testing, which is the split-sample idea from the "Training / Testing /
Validation" slide, and we will need it in Part 4. `export_text` prints the induced tree.

In [ ]:
encoded = cohort[candidates].copy()
encoded["insurance_type"] = encoded["insurance_type"].map(
    {"commercial": 0, "medicare": 1, "medicaid": 2})
encoded["care_network"] = encoded["care_network"].map(
    {"in_network": 0, "community_partner": 1})

X_train, X_test, y_train, y_test = train_test_split(
    encoded, outcome, test_size=0.3, random_state=5200, stratify=outcome)

shallow_tree = DecisionTreeClassifier(max_depth=3, random_state=5200)
shallow_tree.fit(X_train, y_train)

root_feature = candidates[shallow_tree.tree_.feature[0]]
root_threshold = shallow_tree.tree_.threshold[0]
print(f"Root split: {root_feature} <= {root_threshold:.2f}\n")
print(export_text(shallow_tree, feature_names=candidates, decimals=1))

The tree split first on `prior_admissions`, the same feature your mutual information calculation
put at the top. That is not a coincidence and it is not the library being clever: information
gain *is* mutual information, computed on the split rather than on quartile bins. Check the
agreement against `sklearn`'s own estimator below — three different routes, one answer.

In [ ]:
library_mi = pd.Series(
    mutual_info_classif(encoded, outcome, random_state=5200,
                        discrete_features=[c in ("prior_admissions", "insurance_type",
                                                 "care_network") for c in candidates]),
    index=candidates).sort_values(ascending=False)

print("sklearn mutual_info_classif (nats, so the scale differs; the ORDER is the point):")
print(library_mi.round(4).to_string())
hand_rolled_winner = (ranking.iloc[0]["feature"] if ranking["I(X;Y) bits"].max() > 0
                      else "(fill in the Part 2 TODO first)")
print(f"\nHand-rolled winner : {hand_rolled_winner}")
print(f"sklearn winner     : {library_mi.index[0]}")
print(f"Tree's root split  : {root_feature}")

Now read the tree as what it is. Every root-to-leaf path is a conjunction of tests ending in a
conclusion — that is an IF/THEN production rule, the same object you hand-built for the expert
system in Week 6. The difference is provenance, not form: in Week 6 you elicited the rules from
a domain expert, and here they were induced from 840 discharges. Everything the Week 6 notebook
said about conflict resolution, brittleness at the boundaries, and explaining a conclusion to a
clinician applies unchanged to what prints below.

In [ ]:
def tree_to_rules(tree, feature_names):
    """Walk each root-to-leaf path and emit it as an IF/THEN rule, Week 6 style."""
    structure = tree.tree_
    rules = []

    def walk(node, conditions):
        if structure.feature[node] == -2:                     # leaf
            counts = structure.value[node][0]
            predicted = int(np.argmax(counts))
            support = int(structure.n_node_samples[node])
            confidence = counts[predicted] / counts.sum()
            rules.append((conditions, predicted, support, confidence))
            return
        name = feature_names[structure.feature[node]]
        cut = structure.threshold[node]
        walk(structure.children_left[node], conditions + [f"{name} <= {cut:.1f}"])
        walk(structure.children_right[node], conditions + [f"{name} > {cut:.1f}"])

    walk(0, [])
    return rules


for conditions, predicted, support, confidence in tree_to_rules(shallow_tree, candidates):
    verdict = "readmit within 30d" if predicted == 1 else "no readmission"
    print(f"IF   {' AND '.join(conditions)}")
    print(f"THEN {verdict}   (n={support}, confidence {confidence:.2f})\n")

## Part 4 — Let the tree keep splitting

Nothing in tree induction says to stop at depth 3. Take the cap off and the algorithm will keep
finding splits until every leaf is pure, because on any finite sample there is always some
threshold that separates the last two disagreeing patients. Below, fit an unrestricted tree and
compare training accuracy with test accuracy for both trees.

**Predict first:** what will the deep tree's *training* accuracy be? And will its *test*
accuracy go up or down relative to the depth-3 tree?

In [ ]:
deep_tree = DecisionTreeClassifier(random_state=5200)   # no max_depth: split until pure
deep_tree.fit(X_train, y_train)

def score_tree(name, tree):
    return {
        "tree": name,
        "depth": tree.get_depth(),
        "leaves": tree.get_n_leaves(),
        "train acc": round(accuracy_score(y_train, tree.predict(X_train)), 3),
        "test acc": round(accuracy_score(y_test, tree.predict(X_test)), 3),
        "test AUC": round(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]), 3),
    }

print(pd.DataFrame([score_tree("depth 3", shallow_tree),
                    score_tree("unrestricted", deep_tree)]).to_string(index=False))

The deep tree classifies its training set perfectly and is *worse* than the shallow one on
patients it has not seen. It did not learn medicine; it memorized 840 discharges, one leaf at a
time. This is the entire argument for the split-sample and cross-validation slides: training
accuracy is not evidence of anything, because a model with enough capacity can always drive it
to 1.0. Now complete the TODO below to find where the trade-off turns.

In [ ]:
depths_to_try = [1, 2, 3, 4, 6, 8, 12, None]
cv_results = []
for depth in depths_to_try:
    candidate_tree = DecisionTreeClassifier(max_depth=depth, random_state=5200)
    # TODO: replace the training-set score below with 5-fold cross-validated AUC, per the
    # "K-fold cross validation" slide:
    #     scores = cross_val_score(candidate_tree, encoded, outcome, cv=5, scoring="roc_auc")
    #     auc = scores.mean()
    candidate_tree.fit(encoded, outcome)
    auc = roc_auc_score(outcome, candidate_tree.predict_proba(encoded)[:, 1])
    cv_results.append({"max_depth": depth if depth else "none",
                       "AUC": round(auc, 3)})

print(pd.DataFrame(cv_results).to_string(index=False))
print("\nIf AUC just climbs to 1.0 and stays there, you are still scoring the tree on the")
print("very patients it was fit to. That is the mistake the slide is warning about.")

Finally, the picture the "ROC Curve and AUC" slide asks for. One curve per tree, on the held-out
30%. The diagonal is a coin flip. Notice how little of the plot the deep tree's curve occupies
above that diagonal — a tree with two hundred-odd leaves gives you almost no usable ranking of patients by
risk, because nearly every test patient falls into a leaf built from a handful of training cases.

In [ ]:
plt.figure(figsize=(5, 5))
for name, tree in [("depth 3", shallow_tree), ("unrestricted", deep_tree)]:
    fpr, tpr, _ = roc_curve(y_test, tree.predict_proba(X_test)[:, 1])
    auc = roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label=f"{name} (AUC {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Held-out ROC, 30-day readmission")
plt.legend(loc="lower right")
plt.show()

## Talk about it

1. The best single feature removes 0.08 of the 0.92 bits of uncertainty about readmission. If
   you were asked to sign off on deploying this model to assign transitional-care nurses, what
   else would you need to know before that number becomes acceptable or unacceptable?
2. The depth-3 tree is a rule base you can read, and the unrestricted tree is not. At what depth
   does "explainable" stop being true? Is the honest cutoff about depth, or about whether any
   individual rule has enough patients behind it to be worth reading?
3. Mutual information ranked `prior_admissions` first. But this cohort records prior admissions
   less completely for one group of patients. Does mutual information have any way of telling
   you that the top-ranked feature is measured unevenly — and if not, what would?

## What carries forward

The cohort and the ground truth you saw in the generator come back twice. In **Week 11** you
will train a random forest on this same data and ask SHAP and LIME why it made a particular
prediction. In **Week 12** you will audit that model and find the recording gap we planted in
`capture` today, sitting there as a measurable difference in who gets flagged for follow-up.

## Solutions

Completed versions of the two TODOs. These are markdown, not runnable cells, so scrolling here
does not overwrite your work.

**Part 2 — `conditional_entropy`:**

```python
def conditional_entropy(feature, labels):
    feature, labels = np.asarray(feature), np.asarray(labels)
    total = len(labels)
    running = 0.0
    for value in np.unique(feature):
        in_bin = feature == value
        weight = in_bin.sum() / total
        running += weight * entropy(labels[in_bin])
    return running
```

The whole fix is `labels[in_bin]` instead of `labels`: measure the outcome's entropy *inside*
the group of patients the feature picks out, then average those over the groups by size.

**Part 4 — cross-validated AUC:**

```python
depths_to_try = [1, 2, 3, 4, 6, 8, 12, None]
cv_results = []
for depth in depths_to_try:
    candidate_tree = DecisionTreeClassifier(max_depth=depth, random_state=5200)
    scores = cross_val_score(candidate_tree, encoded, outcome, cv=5, scoring="roc_auc")
    cv_results.append({"max_depth": depth if depth else "none",
                       "AUC": round(scores.mean(), 3)})

print(pd.DataFrame(cv_results).to_string(index=False))
```

Cross-validated AUC peaks around depth 3 to 4 near 0.66 and falls to roughly 0.58 for the
unrestricted tree. The training-set version, by contrast, reaches 1.000 and stays there — which
is precisely why nobody should ever report it.